# [LAB-10] 7. [PBT] Titanic Survival (1)

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data
from helpers import *
from IPython.display import display, Markdown

from pandas import DataFrame, crosstab

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 불러오기

In [2]:
origin = load_data("titanic")
origin.head()

📚 타이타닉호 침몰은 역사상 가장 악명 높은 난파 사고 중 하나입니다. 1912년 4월 15일, 첫 항해 중이던 타이타닉호는 빙산과 충돌하여 침몰했습니다. 당시 배에는 모든 승객과 승무원을 태울 수 있는 구명보트가 충분하지 않아 2,224명의 승객과 승무원 중 1,502명이 목숨을 잃었습니다. 생존에는 어느 정도 운이 작용했지만, 일부 집단은 다른 집단보다 생존 가능성이 더 높았던 것으로 보입니다. 이번 과제에서는 승객 데이터(예: 이름, 나이, 성별, 사회경제적 계층 등)를 사용하여 `어떤 유형의 사람들이 생존할 가능성이 더 높았을까?`라는 질문에 답하는 예측 모델을 구축해야 합니다. (출처: https://www.kaggle.com/datasets/yasserh/titanic-dataset)

변수명       설명
-----------  -------------------------------------------------------
PassengerId  탑승객의 ID(인덱스와 같은 개념)
Survived     생존유무(0은 사망 1은 생존)
Pclass       객실의 등급(1=1등급, 2=2등급, 3=3등급)
Name         이름
Sex          성별
SibSp        동승한 형제 혹은 배우자의 수
Parch        동승한 자녀 혹은 부모의 수
Ticket       티켓번호
Fare         요금
Cabin        선실
Embarked     탑승지 (C = Cherbourg, Q = Queenstown, S = Southampton)



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.000,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S


### 3. 컬럼 의미를 정리한 딕셔너리

In [3]:
column_means = {
    "PassengerId": "탑승객 식별자",
    "Survived":    "생존 여부 (0=사망, 1=생존)",
    "Pclass":      "객실 등급 (1=1등급, 2=2등급, 3=3등급)",
    "Name":        "이름",
    "Sex":         "성별",
    "Age":         "나이",
    "SibSp":       "동승한 형제·배우자 수",
    "Parch":       "동승한 부모·자녀 수",
    "Ticket":      "티켓 번호",
    "Fare":        "요금",
    "Cabin":       "선실 번호",
    "Embarked":    "탑승 항구 (C=Cherbourg, Q=Queenstown, S=Southampton)",
}

## #02. 데이터 품질 점검

### 1. 자료형 확인

In [4]:
origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     1309 non-null   int64  
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   str    
 4   Sex          1309 non-null   str    
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    str    
 11  Embarked     1307 non-null   str    
dtypes: float64(2), int64(5), object(1), str(4)
memory usage: 166.2+ KB


### 2. 데이터 중복 점검

In [5]:
df0 = my_qtcheck.check_duplicates(origin, drop=True)

중복된 행의 수: 0


### 3. 불필요한 컬럼 제거와 자료형 변환

| 컬럼 | 제거 이유 |
|---|---|
| `PassengerId` | **행마다 값이 모두 다른 식별자**다. 숫자로 저장되어 있어 그대로 두면 연속형 독립변수로 잘못 투입된다. |
| `Name` | 1,309개가 거의 모두 다른 **자유 텍스트**다. 그대로는 변수가 될 수 없다. |
| `Ticket` | 형식이 제각각인 **문자열 식별자**다. (`A/5 21171`, `113803`, `STON/O2. 3101282` …) |
| `Cabin` | **결측이 1,014건(77.5%)** 이다. 채울 근거가 없는 수준이므로 컬럼째 제거한다. |


- 남은 컬럼 중 `Survived`·`Pclass`·`Sex`·`Embarked`를 **명목형(category)** 으로 변환한다.
  - `Survived`는 종속변수이고, `Pclass`는 1·2·3이라는 숫자로 저장되어 있지만 **"3등급은 1등급의 3배"가 아니므로** 연속형으로 두면 안 된다.

In [6]:
# 원본에서 제거된 중복 데이터가 없으므로 계속 원본 사용
df0 = origin.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df1 = my_qtcheck.set_type(df0, as_category=["Survived", "Pclass", "Sex", "Embarked"])

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Survived  1309 non-null   category
 1   Pclass    1309 non-null   category
 2   Sex       1309 non-null   category
 3   Age       1046 non-null   float64 
 4   SibSp     1309 non-null   int64   
 5   Parch     1309 non-null   int64   
 6   Fare      1308 non-null   float64 
 7   Embarked  1307 non-null   category
dtypes: category(4), float64(2), int64(2)
memory usage: 46.2 KB


### 4. 결측치 점검

In [7]:
my_qtcheck.check_missing_values(df1)

,Missing Count,Missing Ratio (%)
Survived,0,0.000
Pclass,0,0.000
Sex,0,0.000
Age,263,20.092
SibSp,0,0.000
Parch,0,0.000
Fare,1,0.076
Embarked,2,0.153


### 5. 결측치 처리

- 로지스틱 회귀는 결측치가 하나라도 있으면 적합되지 않는다. **모델링 전에 반드시 채워야 한다.**
- 컬럼의 성격에 따라 대체 방법을 다르게 쓴다.

| 컬럼 | 결측 | 대체 방법 | 이유 |
|---|---|---|---|
| `Age` | 263건 (20.1%) | **중앙값** | 오른쪽 꼬리가 있는 분포라 평균은 극단값에 끌려간다 |
| `Fare` | 1건 (0.08%) | **중앙값** | 왜도 4.37로 심하게 치우쳐 있다. 단 1건이라 영향은 미미하다 |
| `Embarked` | 2건 (0.15%) | **최빈값** | 명목형이므로 평균·중앙값을 쓸 수 없다 |

- ⚠️ `Age`의 20.1%를 하나의 값으로 채우는 것은 **가볍게 볼 일이 아니다.**
  나이 분포에 중앙값 자리의 인공적인 봉우리가 생기고, 나이의 효과가 **실제보다 약하게** 추정된다.
  이 한계는 반드시 기록해 두어야 한다.

In [8]:
df2 = my_prep.replace_missing(df1, columns=["Age", "Fare"], method="median")
df3 = my_prep.replace_missing(df2, columns=["Embarked"], method="mode")

# 결측치가 모두 처리되었는지 확인
my_qtcheck.check_missing_values(df3)

결측치 처리 방식: 'median'
컬럼                     결측치                 대체값
--------------------------------------------
Age          263개(20.1%)              28.0
Fare            1개(0.1%)           14.4542
결측치 처리 방식: 'mode'
컬럼                     결측치                 대체값
--------------------------------------------
Embarked        2개(0.2%)                 S


,Missing Count,Missing Ratio (%)
Survived,0,0.000
Pclass,0,0.000
Sex,0,0.000
Age,0,0.000
SibSp,0,0.000
Parch,0,0.000
Fare,0,0.000
Embarked,0,0.000


### 6. 파생변수 생성 — 동승 가족 수 계산

In [9]:
# 결측 처리까지 끝난 데이터에 파생변수를 더한다 (df3 는 그대로 보존)
derived = df3.copy()

# 새로운 파생 변수 생성
derived["FamilySize"] = derived["SibSp"] + derived["Parch"] + 1

# 컬럼 의미 갱신
column_means["FamilySize"] = "동승 가족 수 (SibSp + Parch + 본인)"

# 결과 확인
derived.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize
0,0,3,male,22.000,1,0,7.250,S,2
1,1,1,female,38.000,1,0,71.283,C,2
2,1,3,female,26.000,0,0,7.925,S,1
3,1,1,female,35.000,1,0,53.100,S,2
4,0,3,male,35.000,0,0,8.050,S,1


### 7. 신분 파생변수 추가 (1) - 이름에서 호칭 분리

- 타이타닉의 이름은 `Braund, Mr. Owen Harris`처럼 **`성, 호칭. 이름`** 형식으로 통일되어 있다.
  → **쉼표와 마침표 사이**를 뽑으면 호칭이 나온다. 정규식 `,\s*([^\.]+)\.` 이 그 일을 한다.
- 뽑아 쓰는 대상은 **`origin["Name"]`** 이다. 작업본에서는 이미 지운 컬럼이지만 원본에는 남아 있다.

In [10]:
# ── ① 이름에서 호칭 추출 : "성, 호칭. 이름" 형식이므로 쉼표와 마침표 사이를 뽑는다
title_raw = origin["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
title_raw = title_raw.str.strip()

display(Markdown("**원본 호칭 (통합 전 18종)**"))
display(title_raw.value_counts().rename("빈도").to_frame().T)

**원본 호칭 (통합 전 18종)**

Name,Mr,Miss,Mrs,Master,Rev,Dr,Col,Ms,Major,Mlle,Don,Mme,Lady,Sir,Capt,the Countess,Jonkheer,Dona
빈도,757,260,197,61,8,8,4,2,2,2,1,1,1,1,1,1,1,1


### 8. 신분 파생변수 추가 (2) - 의미별 호칭 정의

In [11]:
# ── 18종의 호칭을 의미가 같은 것끼리 묶어 5종으로 통합한다
#    (도메인 지식이 필요한 부분)
title_map = {
    # 성인 남성 — Don(스페인)·Sir(영국)·Jonkheer(네덜란드)는 남성 귀족 경칭
    "Mr": "Mr", "Don": "Mr", "Sir": "Mr", "Jonkheer": "Mr",
    # 미혼 여성 — Mlle(프랑스)=Miss, Ms=혼인 여부를 밝히지 않는 호칭
    "Miss": "Miss", "Mlle": "Miss", "Ms": "Miss",
    # 기혼 여성 — Mme(프랑스)·Dona(스페인)=Mrs, Lady·the Countess는 여성 귀족 경칭
    "Mrs": "Mrs", "Mme": "Mrs", "Dona": "Mrs", "Lady": "Mrs", "the Countess": "Mrs",
    # 남자아이
    "Master": "Master",
    # 전문직·군인·성직자 — 이름에 직위가 드러난 사람들
    "Dr": "Officer", "Rev": "Officer", "Col": "Officer", "Major": "Officer", "Capt": "Officer",
}

### 9. 신분 파생변수 추가 (3) - 호칭 파생변수 추가 (신분을 의미)

In [12]:
# ── Title 변수 추가 (origin 과 인덱스가 같으므로 그대로 붙는다)
derived["Title"] = title_raw.map(title_map)

# ── 매핑 누락 점검 : 결측이 1건이라도 있으면 title_map 에 빠진 호칭이 있다는 뜻이다
print("Title 매핑 누락 :", derived["Title"].isna().sum(), "건\n")

# ── 문자열이므로 명목형으로 선언한다. 그대로 두면 더미 인코딩 대상이 되지 않는다
derived = my_qtcheck.set_type(derived, as_category=["Title"])

# ── 컬럼 의미 사전도 함께 갱신한다 (이후 EDA 반복문이 이 사전을 참조한다)
column_means["Title"] = "이름에서 추출한 호칭 (Mr/Mrs/Miss/Master/Officer)"

display(origin[["Name"]].join(derived[["Title", "SibSp", "Parch", "FamilySize"]]).head())
display(derived["Title"].value_counts().rename("빈도").to_frame().T)

Title 매핑 누락 : 0 건

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Survived    1309 non-null   category
 1   Pclass      1309 non-null   category
 2   Sex         1309 non-null   category
 3   Age         1309 non-null   float64 
 4   SibSp       1309 non-null   int64   
 5   Parch       1309 non-null   int64   
 6   Fare        1309 non-null   float64 
 7   Embarked    1309 non-null   category
 8   FamilySize  1309 non-null   int64   
 9   Title       1309 non-null   category
dtypes: category(5), float64(2), int64(3)
memory usage: 57.9 KB


,Name,Title,SibSp,Parch,FamilySize
0,"Braund, Mr. Owen Harris",Mr,1,0,2
1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",Mrs,1,0,2
2,"Heikkinen, Miss. Laina",Miss,0,0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,1,0,2
4,"Allen, Mr. William Henry",Mr,0,0,1


Title,Mr,Miss,Mrs,Master,Officer
빈도,760,264,201,61,23


### 10. 생성된 신분 파생변수의 분포 확인

In [13]:
# ── 호칭이 정말 '신분'을 담고 있는지 확인한다
display(Markdown("**호칭별 성별 분포**"))
display(crosstab(derived["Title"], derived["Sex"]))

# ⚠️ 나이는 결측을 채우기 전인 origin 기준으로 본다.
# -> derived 의 Age 는 중앙값 28세로 채워져 있어 호칭별 분포가 왜곡된다.
display(Markdown("**호칭별 나이 요약 (결측 대체 전 · `origin` 기준)**"))
display(origin.assign(Title=derived["Title"])
              .groupby("Title", observed=True)["Age"]
              .agg(["count", "median", "min", "max"]).T)

**호칭별 성별 분포**

Sex,female,male
Title,,
Master,0,61
Miss,264,0
Mr,0,760
Mrs,201,0
Officer,1,22


**호칭별 나이 요약 (결측 대체 전 · `origin` 기준)**

Title,Master,Miss,Mr,Mrs,Officer
count,53.000,213.000,584.000,174.000,22.000
median,4.000,22.000,29.000,35.500,49.500
min,0.330,0.170,11.000,14.000,23.000
max,14.500,63.000,80.000,76.000,70.000


### 11. 명목형 변수의 기술 통계량

In [14]:
cat_desc = my_qtcheck.categorical_summary(derived, value_counts=False)
cat_desc

,Survived,Pclass,Sex,Embarked,Title
count,1309,1309,1309,1309,1309
unique,2,3,2,3,5
top,0,3,male,S,Mr
freq,815,709,843,916,760


### 13. 연속형 변수의 기술 통계량

- 여기서 생성한 기술 통계량 표가 단변량 분석에서 활용되기 때문에, 여기서는 인사이트 발굴 없이 통계량 표만 확인하고 넘어가도 된다.


In [15]:
num_desc = my_qtcheck.numerical_summary(derived)
num_desc.T

,Age,SibSp,Parch,Fare,FamilySize
count,1309.000,1309.000,1309.000,1309.000,1309.000
mean,29.503,0.499,0.385,33.281,1.884
std,12.905,1.042,0.866,51.741,1.584
min,0.170,0.000,0.000,0.000,1.000
25%,22.000,0.000,0.000,7.896,1.000
50%,28.000,0.000,0.000,14.454,1.000
75%,35.000,1.000,0.000,31.275,2.000
max,80.000,8.000,9.000,512.329,11.000
rel_diff,0.054,inf,inf,1.303,0.884
rdiff_flag,similar,large_diff,large_diff,large_diff,large_diff


## #03. 변수 유형 분류와 EDA 범위

### 1. 변수 유형 분류

In [16]:
target = "Survived"    # 종속변수
target_is_continuous = False    # 종속변수 연속형 여부  --> 로지스틱이므로 False
nominal_cols = my_qtcheck.get_categorical_column_names(derived) # 명목형 변수
continuous_cols = my_qtcheck.get_number_column_names(derived)   # 연속형 변수

if target_is_continuous:
    continuous_cols.remove(target)  # 연속형 변수 이름에서 종속변수 항목 제거
else:
    nominal_cols.remove(target)     # 명목형 변수 이름에서 종속변수 항목 제거

print("종속변수 :", target)
print("종속변수 유형: " + ("연속형" if target_is_continuous else "명목형"))
print("명목형   :", nominal_cols)
print("연속형   :", continuous_cols)

종속변수 : Survived
종속변수 유형: 명목형
명목형   : ['Pclass', 'Sex', 'Embarked', 'Title']
연속형   : ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']


### 2. 범주 순서 지정 (서열척도 · 기준 범주)

- 명목형 범주의 순서를 손봐야 하는 경우는 **두 가지**이고, 목적이 서로 다르다.

| 목적 | 대상 | `ordered` | 뜻 |
|---|---|---|---|
| **서열척도 선언** | `Pclass` | `True` | 범주 사이에 **크고 작음이 실제로 있다** |
| **기준 범주 지정** | `Title` | `False` | 순서는 없지만 **더미의 기준을 정하고 싶다** |

In [17]:
# ① 서열척도를 적용할 변수와 순서를 정의한다. (낮은 등급 → 높은 등급)
ordinal_map = {
    "Pclass": [3, 2, 1],    # 3등급 < 2등급 < 1등급
}

# ② 서열은 없지만 더미변수의 기준(reference) 범주를 지정하기 위해 순서만 재배치할 변수
# -> 맨 앞에 둔 범주가 drop_first 로 빠지면서 기준이 된다.
reference_map = {
    "Title": ["Mr", "Officer", "Master", "Miss", "Mrs"],    # 기준: Mr (성인 남성)
}

# 정의된 내용에 따라 순서를 재지정한다.
# -> 데이터셋에 없는 컬럼은 건너뛰므로 다른 데이터셋에서도 오류가 나지 않는다.
df = derived.copy()

for col, order in ordinal_map.items():
    if col in df.columns:
        print(f"'{col}' 컬럼 순서 재지정 (서열척도)   : {order}")
        df[col] = df[col].cat.reorder_categories(order, ordered=True)

for col, order in reference_map.items():
    if col in df.columns:
        print(f"'{col}' 컬럼 순서 재지정 (기준 범주용) : {order}")
        # ordered=False -> 순서만 바꿀 뿐 서열척도로 선언하지는 않는다
        df[col] = df[col].cat.reorder_categories(order, ordered=False)

'Pclass' 컬럼 순서 재지정 (서열척도)   : [3, 2, 1]
'Title' 컬럼 순서 재지정 (기준 범주용) : ['Mr', 'Officer', 'Master', 'Miss', 'Mrs']


### 3. EDA 범위 정리

| 종속유형 | 독립유형 | 분석 유형 | 대상 변수 | 적용 여부 | 비고 |
|---|---|---|---|---|---|
| 연속형 | 연속형 | 상관분석 | — | ❌ 미적용 | 종속변수 타입이 명목형 |
| 연속형 | 명목형(2집단) | T검정 | — | ❌ 미적용 | 종속변수 타입이 명목형 |
| 연속형 | 명목형(3집단+) | ANOVA | — | ❌ 미적용 | 종속변수 타입이 명목형 |
| **명목형(2집단)** | **연속형** | **T검정** | `Age`·`SibSp`·`Parch`·`Fare`·**`FamilySize`** → `Survived` | ✅ **적용** | |
| **명목형(2집단)** | **명목형(2집단)** | **교차분석** | `Sex` → `Survived` | ✅ **적용** | |
| **명목형(2집단)** | **명목형(3집단+)** | **교차분석** | `Pclass`·`Embarked`·**`Title`** → `Survived` | ✅ **적용** | |
| 명목형(3집단+) | 연속형 | ANOVA | — | ❌ 미적용 | 종속변수가 2집단 |
| 명목형(3집단+) | 명목형(2집단) | 교차분석 | — | ❌ 미적용 | 종속변수가 2집단 |
| 명목형(3집단+) | 명목형(3집단+) | 교차분석 | — | ❌ 미적용 | 종속변수가 2집단 |